# Phase 5 — Error Analysis

**Course**: CS5143 Natural Language Processing — Spring 2026, FAST-NUCES  
**Student**: Muhammad Azhar (24K-7606)

This notebook identifies and categorises model errors on the English and Spanish test sets:
- **False positives (FP)**: model predicts an entity where gold is `O`
- **False negatives (FN)**: model predicts `O` where gold is an entity
- **Type confusion**: model predicts the wrong entity type (boundary correct, type wrong)

**Prerequisites**: Complete `04_evaluation.ipynb` — predictions will be re-run here.

## 1. Setup

In [ ]:
import sys
import os
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_from_disk
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
)

from dataset import id2label, load_conll
from utils import tokenizer, get_data_paths

BEST_MODEL_PATH = REPO_ROOT / 'outputs' / 'checkpoints' / 'best_model'

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(
        f'Best model not found at {BEST_MODEL_PATH}.\n'
        'Complete Phase 4 training first.'
    )

model = AutoModelForTokenClassification.from_pretrained(str(BEST_MODEL_PATH))
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print('Model loaded.')

## 2. Collect Errors

We re-run inference on both test sets to extract token-level mismatches.
Raw sentences are loaded alongside predictions so error examples include context.

In [ ]:
def get_predictions(lang, split='test'):
    """Return (sentences, true_labels, true_preds) for a language split."""
    train_f, dev_f, test_f = get_data_paths(lang)
    split_file = {'train': train_f, 'validation': dev_f, 'test': test_f}[split]
    sentences = load_conll(split_file)

    ds = load_from_disk(str(REPO_ROOT / 'data' / 'processed' / lang))
    preds_out = trainer.predict(ds[split])
    logits, labels = preds_out.predictions, preds_out.label_ids
    predictions = np.argmax(logits, axis=2)

    true_labels = [
        [id2label[l] for l in seq if l != -100]
        for seq in labels
    ]
    true_preds = [
        [id2label[p] for (p, l) in zip(ps, ls) if l != -100]
        for ps, ls in zip(predictions, labels)
    ]
    return sentences, true_labels, true_preds


def collect_errors(sentences, true_labels, true_preds):
    """Collect FP, FN, and type-confusion errors at the token level."""
    errors = {'false_positive': [], 'false_negative': [], 'type_confusion': []}
    for sent, ts, ps in zip(sentences, true_labels, true_preds):
        tokens = sent['tokens']
        context = ' '.join(tokens)
        for i, (t, p) in enumerate(zip(ts, ps)):
            token = tokens[i] if i < len(tokens) else ''
            if t == 'O' and p != 'O':
                errors['false_positive'].append({
                    'token': token,
                    'true':  t,
                    'pred':  p,
                    'context': context,
                })
            elif t != 'O' and p == 'O':
                errors['false_negative'].append({
                    'token': token,
                    'true':  t,
                    'pred':  p,
                    'context': context,
                })
            elif t != 'O' and p != 'O' and t != p:
                errors['type_confusion'].append({
                    'token': token,
                    'true':  t,
                    'pred':  p,
                    'context': context,
                })
    return errors


print('Running inference on EN test set ...')
en_sents, en_true, en_pred = get_predictions('en', 'test')
en_errors = collect_errors(en_sents, en_true, en_pred)

print('Running inference on ES test set ...')
es_sents, es_true, es_pred = get_predictions('es', 'test')
es_errors = collect_errors(es_sents, es_true, es_pred)

print('Error collection complete.')

## 3. Error Breakdown

Total count of each error category for English and Spanish.

In [ ]:
breakdown = pd.DataFrame({
    'Error Type':    ['False Positive', 'False Negative', 'Type Confusion'],
    'EN Count':      [len(en_errors['false_positive']),
                      len(en_errors['false_negative']),
                      len(en_errors['type_confusion'])],
    'ES Count':      [len(es_errors['false_positive']),
                      len(es_errors['false_negative']),
                      len(es_errors['type_confusion'])],
})
print('Error counts by category:')
display(breakdown)

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(breakdown))
w = 0.35
ax.bar([i - w/2 for i in x], breakdown['EN Count'], width=w, label='EN', color='#4C72B0')
ax.bar([i + w/2 for i in x], breakdown['ES Count'], width=w, label='ES', color='#DD8452')
ax.set_xticks(list(x))
ax.set_xticklabels(breakdown['Error Type'])
ax.set_ylabel('Token count')
ax.set_title('Error Category Breakdown — EN vs ES')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(str(REPO_ROOT / 'outputs' / 'results' / 'error_breakdown.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Sample Errors

Inspect concrete examples to understand failure patterns.

In [ ]:
def show_errors(errors, category, n=10, lang='EN'):
    items = errors[category][:n]
    if not items:
        print(f'No {category} errors found.')
        return
    rows = [{'Token': e['token'], 'True': e['true'], 'Pred': e['pred'],
             'Context (truncated)': e['context'][:80]} for e in items]
    df = pd.DataFrame(rows)
    print(f'\n{lang} — {category.replace("_", " ").title()} (first {len(items)}):')
    display(df)


show_errors(en_errors, 'false_negative',  n=10, lang='EN')
show_errors(en_errors, 'false_positive',  n=10, lang='EN')
show_errors(en_errors, 'type_confusion',  n=10, lang='EN')

## 5. EN vs ES Comparison

Comparing error rates between English and Spanish reveals language-specific challenges
and the quality of cross-lingual transfer.

In [ ]:
show_errors(es_errors, 'false_negative',  n=10, lang='ES')
show_errors(es_errors, 'false_positive',  n=10, lang='ES')
show_errors(es_errors, 'type_confusion',  n=10, lang='ES')

In [ ]:
# Normalise by total entity tokens in each test set
def total_entity_tokens(label_seqs):
    return sum(1 for seq in label_seqs for l in seq if l != 'O')

en_entity_total = total_entity_tokens(en_true)
es_entity_total = total_entity_tokens(es_true)

rate_df = pd.DataFrame({
    'Error Type':   ['False Positive', 'False Negative', 'Type Confusion'],
    'EN Rate':      [
        round(len(en_errors['false_positive']) / max(en_entity_total, 1), 4),
        round(len(en_errors['false_negative']) / max(en_entity_total, 1), 4),
        round(len(en_errors['type_confusion']) / max(en_entity_total, 1), 4),
    ],
    'ES Rate':      [
        round(len(es_errors['false_positive']) / max(es_entity_total, 1), 4),
        round(len(es_errors['false_negative']) / max(es_entity_total, 1), 4),
        round(len(es_errors['type_confusion']) / max(es_entity_total, 1), 4),
    ],
})

print('Error rates (normalised by entity token count):')
display(rate_df)

## Summary

**What was done in this phase:**
- Categorised token-level errors into false positives, false negatives, and type confusion
- Inspected 10 concrete examples of each category for both EN and ES
- Compared normalised error rates between languages to assess cross-lingual transfer quality

**Typical findings to discuss:**
- FNs often occur for rare clinical terms and abbreviations not covered by training data
- FPs tend to cluster around common medical nouns that lack clear entity context
- Type confusion (e.g., DIS ↔ SYM) reflects semantic overlap between disease and symptom descriptions
- ES may show higher FN rates if the training set is smaller or has different entity density

**Next step**: Phase 6 — Final Notebook & Submission (`MultiClinNER_XLM-R.ipynb`)